# 04 — Обучение uplift T-learner (treatment + control)

Цель:
- обучить две модели:
  - p(click | treatment = 1)
  - p(click | treatment = 0)
- сохранить их в `models/`,
- получить базовые метрики качества (AUC) для обеих частей T-learner.

In [1]:
import sys
from pathlib import Path

import numpy as np
import pandas as pd

from sklearn.model_selection import train_test_split
from sklearn.metrics import roc_auc_score

from catboost import CatBoostClassifier

PROJECT_ROOT = Path("..").resolve()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.append(str(PROJECT_ROOT))

from src.data_prep.feature_engineering import load_ml_dataset, prepare_features
from src.models.uplift_treatment import save_treatment_model
from src.models.uplift_control import save_control_model
from src.utils.config import MODELS_DIR

In [2]:
df = load_ml_dataset()
print("ML dataset shape:", df.shape)
df.head()

ML dataset shape: (15307, 35)


,client_id,treatment,offer_id,treatment_date,offer_type,offer_category,cost,offer_AOV,channel,conversion,...,age,gender,price_segment,treatment_dow,treatment_month,treatment_hour,time_morning,time_afternoon,time_evening,time_night
0,17850,1,0,2024-04-12,discount_10,category_0,2.09,18.15,app,0,...,37,F,budget,4,4,0,0,0,0,1
1,17850,0,2,2024-06-28,free_delivery,category_0,3.84,18.15,app,0,...,50,F,budget,4,6,0,0,0,0,1
2,17850,0,2,2024-04-02,free_delivery,category_0,1.44,18.15,web,0,...,39,F,budget,1,4,0,0,0,0,1
3,17850,0,1,2024-01-15,discount_20,category_0,5.21,18.15,web,0,...,22,F,budget,0,1,0,0,0,0,1
4,13047,0,1,2024-04-16,discount_20,category_8,2.00,18.82,web,0,...,60,M,budget,1,4,0,0,0,0,1


In [3]:
df_t = df[df["treatment"] == 1].copy()
df_c = df[df["treatment"] == 0].copy()

print("Treatment rows:", df_t.shape, "conversion mean:", df_t["conversion"].mean())
print("Control rows:", df_c.shape, "conversion mean:", df_c["conversion"].mean())

X_t, y_t, ids_t, meta_treat = prepare_features(df_t)
X_c, y_c, ids_c, meta_control = prepare_features(df_c)

CAT_FEATURES_T = [c for c in meta_treat["categorical_features"] if c in X_t.columns]
CAT_FEATURES_C = [c for c in meta_control["categorical_features"] if c in X_c.columns]

print("Cat features (treat):", CAT_FEATURES_T)
print("Cat features (control):", CAT_FEATURES_C)

print("X_t:", X_t.shape, "y_t mean:", float(y_t.mean()))
print("X_c:", X_c.shape, "y_c mean:", float(y_c.mean()))

Treatment rows: (12319, 35) conversion mean: 0.2964526341423817
Control rows: (2988, 35) conversion mean: 0.09303882195448461
Cat features (treat): ['offer_type', 'offer_category', 'channel', 'favorite_category', 'visited_category_14d', 'category_affinity_top1', 'gender', 'price_segment']
Cat features (control): ['offer_type', 'offer_category', 'channel', 'favorite_category', 'visited_category_14d', 'category_affinity_top1', 'gender', 'price_segment']
X_t: (12319, 35) y_t mean: 0.2964526341423817
X_c: (2988, 35) y_c mean: 0.09303882195448461


In [4]:
X_tr_t, X_val_t, y_tr_t, y_val_t = train_test_split(
    X_t, y_t, test_size=0.25, random_state=42, stratify=y_t
)

X_tr_c, X_val_c, y_tr_c, y_val_c = train_test_split(
    X_c, y_c, test_size=0.25, random_state=42, stratify=y_c
)

print("Train/Val treat:", X_tr_t.shape, X_val_t.shape)
print("Train/Val control:", X_tr_c.shape, X_val_c.shape)

Train/Val treat: (9239, 35) (3080, 35)
Train/Val control: (2241, 35) (747, 35)


In [5]:
COMMON_PARAMS = dict(
    iterations=500,
    depth=6,
    learning_rate=0.05,
    loss_function="Logloss",
    eval_metric="AUC",
    random_seed=42,
    verbose=100,
)

treatment_params = COMMON_PARAMS.copy()
control_params = COMMON_PARAMS.copy()
control_params["random_seed"] = 43

In [6]:
model_treat = CatBoostClassifier(**treatment_params)
model_treat.fit(
    X_tr_t, y_tr_t,
    eval_set=(X_val_t, y_val_t),
    cat_features=CAT_FEATURES_T,
    use_best_model=True,
)

p_val_t = model_treat.predict_proba(X_val_t)[:, 1]
auc_t = roc_auc_score(y_val_t, p_val_t)

print("Treatment AUC:", auc_t)

0:	test: 0.7173792	best: 0.7173792 (0)	total: 68ms	remaining: 33.9s
100:	test: 0.7343262	best: 0.7343262 (100)	total: 632ms	remaining: 2.5s
200:	test: 0.7352456	best: 0.7357363 (181)	total: 1.23s	remaining: 1.83s
300:	test: 0.7332167	best: 0.7359870 (252)	total: 1.87s	remaining: 1.24s
400:	test: 0.7307567	best: 0.7359870 (252)	total: 2.5s	remaining: 619ms
499:	test: 0.7264185	best: 0.7359870 (252)	total: 3.14s	remaining: 0us

bestTest = 0.7359870324
bestIteration = 252

Shrink model to first 253 iterations.
Treatment AUC: 0.7359870324103815


In [7]:
model_control = CatBoostClassifier(**control_params)
model_control.fit(
    X_tr_c, y_tr_c,
    eval_set=(X_val_c, y_val_c),
    cat_features=CAT_FEATURES_C,
    use_best_model=True,
)

p_val_c = model_control.predict_proba(X_val_c)[:, 1]
auc_c = roc_auc_score(y_val_c, p_val_c)

print("Control AUC:", auc_c)

0:	test: 0.5518576	best: 0.5518576 (0)	total: 3.1ms	remaining: 1.54s
100:	test: 0.4913642	best: 0.5518576 (0)	total: 221ms	remaining: 874ms
200:	test: 0.4890342	best: 0.5518576 (0)	total: 494ms	remaining: 735ms
300:	test: 0.5192382	best: 0.5518576 (0)	total: 779ms	remaining: 515ms
400:	test: 0.5423240	best: 0.5518576 (0)	total: 1.07s	remaining: 264ms
499:	test: 0.5423453	best: 0.5518576 (0)	total: 1.36s	remaining: 0us

bestTest = 0.551857552
bestIteration = 0

Shrink model to first 1 iterations.
Control AUC: 0.5518575520499338


In [8]:
n = min(len(p_val_t), len(p_val_c))
uplift_val = p_val_t[:n] - p_val_c[:n]

pd.Series(uplift_val).describe()

count    747.000000
mean      -0.174696
std        0.176134
min       -0.416705
25%       -0.309898
50%       -0.261760
75%       -0.004251
max        0.285562
dtype: float64

In [9]:
MODELS_DIR.mkdir(parents=True, exist_ok=True)

save_treatment_model(model_treat, meta_treat)
save_control_model(model_control, meta_control)

print("Saved models to:", MODELS_DIR)
print("Treatment meta keys:", meta_treat.keys())
print("Control meta keys:", meta_control.keys())

Saved models to: /Users/rustamakhmedzianov/Documents/Projects/aero_nbo_uplift/models
Treatment meta keys: dict_keys(['numeric_features', 'categorical_features', 'feature_cols'])
Control meta keys: dict_keys(['numeric_features', 'categorical_features', 'feature_cols'])


In [10]:
from src.models.uplift_treatment import load_treatment_model
from src.models.uplift_control import load_control_model

m_t, mt = load_treatment_model()
m_c, mc = load_control_model()

print("Reload OK.")
print("treat feature_cols:", len(mt["feature_cols"]))
print("control feature_cols:", len(mc["feature_cols"]))

Reload OK.
treat feature_cols: 35
control feature_cols: 35


In [11]:
from src.models.uplift_scoring import add_uplift_scores

df_small = df.sample(5, random_state=42).copy()
df_scored = add_uplift_scores(df_small)

df_scored[["client_id", "offer_id", "p_treat", "p_control", "uplift", "expected_gain_uplift", "conversion", "treatment"]]

,client_id,offer_id,p_treat,p_control,uplift,expected_gain_uplift,conversion,treatment
9892,16614,0,0.188968,0.468856,-0.279888,-7.259417,0,1
8839,13835,1,0.478707,0.473848,0.004858,-2.606301,0,0
4397,16755,1,0.134603,0.467461,-0.332858,-4.989149,0,1
3477,14669,1,0.120553,0.468856,-0.348303,-4.741821,0,1
12124,14306,2,0.239152,0.475300,-0.236149,-19.888903,0,0


## Итоги ноутбука

1. Обучена treatment-модель `p(click | treatment = 1)` на CatBoost:
   - AUC по валидации ≈ `auc_t`.
2. Для control-группы `treatment = 0` обнаружено, что таргет `conversion`
   содержит только один класс → полноценную ML-модель обучить нельзя.
   - Введён фолбек: `DummyClassifier`, который предсказывает константу (в нашем случае 0).
   - В этом сценарии `p_control ≈ 0`, а uplift ≈ `p_treat`.
3. Обе модели и метаинформация сохранены в:
   - `models/uplift_treatment.cbm` (+ meta),
   - `models/uplift_control.pkl` или аналогичный файл (+ meta).
4. Быстрый тест на сэмпле показывает:
   - адекватные вероятности `p_treat`,
   - `p_control` близкое к нулю для DummyClassifier,
   - ненулевой uplift = `p_treat - p_control`.

Дальше:
- реализуем модуль `src/models/uplift_scoring.py` (подсчёт `p_treat`, `p_control`, uplift, expected_gain и выбор офферов),
- сравним uplift-стратегию с rule-based baseline в `05_uplift_vs_rule_based.ipynb`,
- затем обернём всё в FastAPI-сервис.